# Adding the MD&A Text

The MD&A (Item 7 of the 10-K, management's discussion and analysis) of every 10-K: from EDGAR-CORPUS for fiscal years 2008 to 2020 when it passes the check against future text (`collect_mdna_corpus.py`, saved in `data/mdna_corpus/`), and otherwise cut out of the SEC filing by our own code (`collect_mdna.py`, saved in `data/mdna_sec/`) and cleaned so it stops where the MD&A ends (`clean_mdna_sec.py`, saved in `data/mdna_sec_clean/`). This notebook turns it into features and tests them with the same walk forward test as `sota_model.ipynb`.

**Which text a report uses.** A 10-K uses its own MD&A. A 10-Q has no MD&A of this kind, so it uses the company's latest 10-K filed on or before the 10-Q. Nothing filed after the report is used.

**Text features.**
1. **TF-IDF**: how often each word and two-word phrase appears in the MD&A, weighted down when it appears in most MD&As. The words are chosen from 10-Ks filed up to 2014, before the test years.
2. **LSA**: the TF-IDF squeezed into 100 numbers per MD&A (truncated SVD, also fitted on 10-Ks up to 2014). These go into XGBoost next to the other features.
3. **Text score**: a logistic regression on the TF-IDF, trained in the walk forward like XGBoost, that gives each report a bankruptcy score from its text alone (like the TF-IDF model of Arno et al., 2023). It is combined with XGBoost by averaging their ranks, with the weight chosen on 2013 and 2014.

**Feature sets**, to see what each source adds:
- **10-K and 10-Q**: the numbers, the filing facts and the phrases in the reports
- **+ NT**: plus late filing notices (NT 10-K, NT 10-Q)
- **+ 8-K**: plus the warning 8-Ks filed before the report
- each of these **+ MD&A text**

In [1]:
import glob
import gzip
import json
import os
import time

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

pd.set_option("display.max_rows", 150)
everything = pd.read_pickle("long_dataframe_2009_to_2026.pkl")
groups = json.load(open("feature_groups.json"))
df = everything[everything["label_known"] & ~everything["already_bankrupt"]].reset_index(drop=True)
df["year"] = df["filed_date"].dt.year

#the XGBoost settings Optuna chose in sota_model.ipynb, section 6 (tuned on 2009 to 2014)
XGB_PARAMS = {"n_estimators": 1400, "learning_rate": 0.030435151413863663, "max_depth": 8, "min_child_weight": 2.649496317267566,
              "subsample": 0.8794818753946948, "colsample_bytree": 0.6094118179765666, "gamma": 0.43319316787210643,
              "reg_alpha": 0.0015962113607134825, "reg_lambda": 5.696284007633162, "max_delta_step": 1,
              "scale_pos_weight": 6.61283698692129}
print(df.shape)

(364407, 183)


## The Text Each Report Uses

In [2]:
#corpus text where it passed the check, otherwise our own cut of the SEC filing
mdna_files = {os.path.basename(p)[:-7]: p for p in glob.glob("data/mdna_sec_clean/*.txt.gz")}
corpus_files = {os.path.basename(p)[:-7]: p for p in glob.glob("data/mdna_corpus/*.txt.gz")}
mdna_files.update(corpus_files)
text_source = pd.Series({a: "EDGAR-CORPUS" if a in corpus_files else "SEC, our cut" for a in mdna_files})
ten_k = everything.loc[everything["form"].str.startswith("10-K") & everything["adsh"].isin(mdna_files), ["adsh", "cik", "filed_date"]]
ten_k = ten_k.rename(columns={"adsh": "text_adsh", "filed_date": "text_date"}).sort_values("text_date")

#each report gets the latest 10-K with an MD&A filed on or before it, by the same company (a 10-K gets its own)
order = df[["cik", "filed_date"]].reset_index().sort_values("filed_date")
matched = pd.merge_asof(order, ten_k, left_on="filed_date", right_on="text_date", by="cik", direction="backward")
df["text_adsh"] = matched.set_index("index")["text_adsh"].reindex(df.index).values
df["text_age_days"] = (df["filed_date"] - matched.set_index("index")["text_date"].reindex(df.index)).dt.days.values

docs = sorted(df["text_adsh"].dropna().unique())
print("MD&A files:", len(mdna_files), " used by reports:", len(docs))
print("share of reports with a text:", round(df["text_adsh"].notna().mean(), 3))
print(df.groupby(df["form"].str[:4])["text_adsh"].apply(lambda s: s.notna().mean()).round(3))

MD&A files: 85969  used by reports: 85219
share of reports with a text: 0.91
form
10-K    0.985
10-Q    0.886
Name: text_adsh, dtype: float64


## TF-IDF and LSA

Each MD&A is cut to its first 40,000 characters (about 6,000 words, most of a typical MD&A) to keep the memory use small. The vocabulary (20,000 words and two-word phrases that appear in at least 20 MD&As and in at most half of them, without common English words) and the LSA are fitted on 10-Ks filed up to 2014.

In [3]:
def read_text(adsh, limit=40000):
    with gzip.open(mdna_files[adsh], "rt") as f:
        return f.read(limit)


start = time.time()
doc_year = everything.set_index("adsh").loc[docs, "filed_date"].dt.year
fit_docs = [a for a in docs if doc_year[a] <= 2014]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=20, max_df=0.5, max_features=20000, stop_words="english",
                             sublinear_tf=True, dtype=np.float32)
vectorizer.fit(read_text(a) for a in fit_docs)

#every MD&A, in batches so the texts aren't all in memory at once
from scipy import sparse
parts = []
for i in range(0, len(docs), 5000):
    parts.append(vectorizer.transform(read_text(a) for a in docs[i:i + 5000]))
tfidf = sparse.vstack(parts).tocsr()
row_of = {a: i for i, a in enumerate(docs)}

svd = TruncatedSVD(n_components=100, random_state=42).fit(tfidf[[row_of[a] for a in fit_docs]])
lsa = svd.transform(tfidf).astype(np.float32)
lsa_names = [f"text_lsa_{i}" for i in range(lsa.shape[1])]
has_text = df["text_adsh"].notna().values
lsa_rows = np.full((len(df), lsa.shape[1]), np.nan, dtype=np.float32)
lsa_rows[has_text] = lsa[[row_of[a] for a in df.loc[has_text, "text_adsh"]]]
df[lsa_names] = lsa_rows
print("fitted on", len(fit_docs), "10-Ks, tfidf", tfidf.shape, "took", round((time.time() - start) / 60, 1), "minutes")

fitted on 21985 10-Ks, tfidf (85219, 20000) took 5.6 minutes


/var/folders/79/fbgym5f51g38htczmg5vvbh00000gp/T/ipykernel_77425/3610264993.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lsa_names] = lsa_rows
/var/folders/79/fbgym5f51g38htczmg5vvbh00000gp/T/ipykernel_77425/3610264993.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[lsa_names] = lsa_rows
/var/folders/79/fbgym5f51g38htczmg5vvbh00000gp/T/ipykernel_77425/3610264993.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

## The Walk Forward Test

The same as `sota_model.ipynb`: one model for every two test years, trained on reports filed 2 or more years before the first of them (their answers were known), testing every report of the two years. `summary` gives the PR-AUC with a 90% interval from 300 bootstrap draws of the companies, and the change against another model on the same draws.

In [4]:
TEST_YEARS = list(range(2016, 2026))


def xgb_model(seed=42):
    return XGBClassifier(**XGB_PARAMS, tree_method="hist", max_bin=128, eval_metric="aucpr", n_jobs=10, random_state=seed)


def walk_forward(features):
    parts = []
    for year in TEST_YEARS[::2]:
        fit = df[df["year"] <= year - 2]
        test = df[df["year"].isin([year, year + 1])]
        scores = xgb_model().fit(fit[features], fit["Bankruptcy"]).predict_proba(test[features])[:, 1]
        parts.append(pd.DataFrame({"score": scores, "year": test["year"].values}, index=test.index))
    return pd.concat(parts)


def text_model():
    return LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000, solver="liblinear")


def text_walk_forward(years=TEST_YEARS, fit_up_to=None):
    #the text score: trained on the 10-Ks of the training reports (each MD&A once), scores every test report by its text
    parts = []
    for year in years[::2]:
        fit_10k = df[(df["year"] <= (fit_up_to or year - 2)) & df["form"].str.startswith("10-K") & df["text_adsh"].notna()]
        model = text_model().fit(tfidf[[row_of[a] for a in fit_10k["text_adsh"]]], fit_10k["Bankruptcy"])
        test = df[df["year"].isin([year, year + 1]) & df["year"].isin(years)]
        scores = np.full(len(test), np.nan)
        ok = test["text_adsh"].notna().values
        scores[ok] = model.predict_proba(tfidf[[row_of[a] for a in test.loc[ok, "text_adsh"]]])[:, 1]
        parts.append(pd.DataFrame({"score": scores, "year": test["year"].values}, index=test.index))
    return pd.concat(parts)


def rank_in_year(scores):
    #reports without a text get the middle rank
    return scores.groupby("year")["score"].rank(pct=True).fillna(0.5)


def blend(numbers, text, weight):
    out = numbers[["year"]].copy()
    out["score"] = (1 - weight) * rank_in_year(numbers) + weight * rank_in_year(text.reindex(numbers.index).assign(year=numbers["year"]))
    return out


def summary(scores, name="", against=None, years=None, ten_k_only=False, n_boot=300):
    rows = scores if years is None else scores[scores["year"].isin(years)]
    if ten_k_only:
        rows = rows[df.loc[rows.index, "form"].str.startswith("10-K").values]
    y, s = df.loc[rows.index, "Bankruptcy"].values, rows["score"].fillna(rows["score"].median()).values
    row = {"model": name, "reports": len(y), "bankruptcies": int(y.sum()), "ROC-AUC": roc_auc_score(y, s),
           "PR-AUC": average_precision_score(y, s), "random PR-AUC": y.mean()}
    row["times random"] = row["PR-AUC"] / row["random PR-AUC"]
    cik = df.loc[rows.index, "cik"].values
    companies = np.unique(cik)
    rows_of = pd.Series(np.arange(len(cik))).groupby(cik).apply(np.array)
    other = None if against is None else against.loc[rows.index, "score"].fillna(against["score"].median()).values
    rng, boot, change = np.random.default_rng(0), [], []
    for _ in range(n_boot):
        pick = np.concatenate(rows_of.loc[rng.choice(companies, len(companies))].values)
        if y[pick].sum() > 0:
            boot.append(average_precision_score(y[pick], s[pick]))
            if other is not None:
                change.append(boot[-1] - average_precision_score(y[pick], other[pick]))
    row["PR-AUC low (5%)"], row["PR-AUC high (95%)"] = np.percentile(boot, [5, 95])
    if other is not None:
        row["change"] = row["PR-AUC"] - average_precision_score(y, other)
        row["change low (5%)"], row["change high (95%)"] = np.percentile(change, [5, 95])
    return row


WINDOWS = {"2016-2021, 10-K only (Arno et al. test)": dict(years=range(2016, 2022), ten_k_only=True),
           "2016-2021, all reports": dict(years=range(2016, 2022)),
           "2023-2025, all reports (our years)": dict(years=range(2023, 2026))}


def window_table(named_scores, against_name=None):
    rows = []
    for window, kwargs in WINDOWS.items():
        for name, scores in named_scores.items():
            against = None if against_name in (None, name) else named_scores[against_name]
            rows.append({"window": window, **summary(scores, name, against=against, **kwargs)})
    return pd.DataFrame(rows).set_index(["window", "model"])

## The Weight of the Text Score

Chosen on 2013 and 2014 only (models trained on reports filed up to 2011 and 2012), so the test years aren't used to pick it.

In [5]:
warning = groups["warning signs"]
eight_k = [f for f in warning if f.startswith(("item_", "days_since_item_"))]
nt = [f for f in warning if "late_filing_notice" in f]
reports_only = sum([groups[g] for g in groups if g != "warning signs"], []) + [f for f in warning if f not in eight_k + nt]
feature_sets = {"10-K and 10-Q": reports_only, "+ NT": reports_only + nt, "+ NT + 8-K": reports_only + nt + eight_k}
print({k: len(v) for k, v in feature_sets.items()}, " 8-K:", len(eight_k), " NT:", len(nt))

start = time.time()
val = df[df["year"].isin([2013, 2014])]
val_parts_num, val_parts_text = [], []
for year in [2013]:
    fit = df[df["year"] <= year - 2]
    test = df[df["year"].isin([year, year + 1])]
    features = feature_sets["+ NT"] + lsa_names
    val_parts_num.append(pd.DataFrame({"score": xgb_model().fit(fit[features], fit["Bankruptcy"]).predict_proba(test[features])[:, 1],
                                       "year": test["year"].values}, index=test.index))
val_num = pd.concat(val_parts_num)
val_text = text_walk_forward(years=[2013, 2014])
weights = {}
for w in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    b = blend(val_num, val_text, w)
    weights[w] = average_precision_score(df.loc[b.index, "Bankruptcy"], b["score"])
TEXT_WEIGHT = max(weights, key=weights.get)
print("validation PR-AUC by text weight:", {k: round(v, 3) for k, v in weights.items()}, " chosen:", TEXT_WEIGHT,
      " took", round((time.time() - start) / 60, 1), "minutes")

{'10-K and 10-Q': 128, '+ NT': 130, '+ NT + 8-K': 140}  8-K: 10  NT: 2


validation PR-AUC by text weight: {0: np.float64(0.059), 0.1: np.float64(0.048), 0.2: np.float64(0.04), 0.3: np.float64(0.034), 0.4: np.float64(0.029), 0.5: np.float64(0.024), 0.6: np.float64(0.021)}  chosen: 0  took 0.1 minutes


## Results

In [6]:
start = time.time()
text_scores = text_walk_forward()
results = {"MD&A text alone": text_scores}
for name, features in feature_sets.items():
    results[name] = walk_forward(features)
    with_text = walk_forward(features + lsa_names)
    results[f"{name} + MD&A text"] = blend(with_text, text_scores, TEXT_WEIGHT)
    print(name, "done", round((time.time() - start) / 60, 1), "minutes", flush=True)
table = window_table(results, against_name="10-K and 10-Q")
table[["reports", "bankruptcies", "ROC-AUC", "PR-AUC", "times random", "PR-AUC low (5%)", "PR-AUC high (95%)",
       "change", "change low (5%)", "change high (95%)"]].round(3)

10-K and 10-Q done 4.6 minutes


+ NT done 8.1 minutes


+ NT + 8-K done 11.9 minutes


reports  \
window                                  model                                
2016-2021, 10-K only (Arno et al. test) MD&A text alone              34753   
                                        10-K and 10-Q                34753   
                                        10-K and 10-Q + MD&A text    34753   
                                        + NT                         34753   
                                        + NT + MD&A text             34753   
                                        + NT + 8-K                   34753   
                                        + NT + 8-K + MD&A text       34753   
2016-2021, all reports                  MD&A text alone             141743   
                                        10-K and 10-Q               141743   
                                        10-K and 10-Q + MD&A text   141743   
                                        + NT                        141743   
                                        + NT + MD&A text            141743   
                                        + NT + 8-K                  141743   
                                        + NT + 8-K + MD&A text      141743   
2023-2025, all reports (our years)      MD&A text alone              63767   
                                        10-K and 10-Q                63767   
                                        10-K and 10-Q + MD&A text    63767   
                                        + NT                         63767   
                                        + NT + MD&A text             63767   
                                        + NT + 8-K                   63767   
                                        + NT + 8-K + MD&A text       63767   

                                                                   bankruptcies  \
window                                  model                                     
2016-2021, 10-K only (Arno et al. test) MD&A text alone                     268   
                                        10-K and 10-Q                       268   
                                        10-K and 10-Q + MD&A text           268   
                                        + NT                                268   
                                        + NT + MD&A text                    268   
                                        + NT + 8-K                          268   
                                        + NT + 8-K + MD&A text              268   
2016-2021, all reports                  MD&A text alone                    1036   
                                        10-K and 10-Q                      1036   
                                        10-K and 10-Q + MD&A text          1036   
                                        + NT                               1036   
                                        + NT + MD&A text                   1036   
                                        + NT + 8-K                         1036   
                                        + NT + 8-K + MD&A text             1036   
2023-2025, all reports (our years)      MD&A text alone                     600   
                                        10-K and 10-Q                       600   
                                        10-K and 10-Q + MD&A text           600   
                                        + NT                                600   
                                        + NT + MD&A text                    600   
                                        + NT + 8-K                          600   
                                        + NT + 8-K + MD&A text              600   

                                                                   ROC-AUC  \
window                                  model                                
2016-2021, 10-K only (Arno et al. test) MD&A text alone              0.818   
                                        10-K and 10-Q                0.869   
                                        10-K and 1

## Against the Best Published Result

Arno et al. (2023), same test years (2016 to 2021) and 10-Ks only: 0.156 with numbers only, 0.264 with the MD&A text (their share of 1s: 0.67%). "Times random" is the fairest comparison between tests with a different share of 1s.

In [7]:
arno = pd.DataFrame([{"model": "Arno et al. (2023), numbers only", "ROC-AUC": 0.936, "PR-AUC": 0.156, "random PR-AUC": 0.0067},
                     {"model": "Arno et al. (2023), numbers + MD&A text", "ROC-AUC": 0.948, "PR-AUC": 0.264, "random PR-AUC": 0.0067}])
arno["times random"] = arno["PR-AUC"] / arno["random PR-AUC"]
ours = table.xs("2016-2021, 10-K only (Arno et al. test)", level="window").reset_index()
pd.concat([arno, ours[["model", "ROC-AUC", "PR-AUC", "random PR-AUC", "times random"]]]).set_index("model").round(3)

,ROC-AUC,PR-AUC,random PR-AUC,times random
model,,,,
"Arno et al. (2023), numbers only",0.936,0.156,0.007,23.284
"Arno et al. (2023), numbers + MD&A text",0.948,0.264,0.007,39.403
MD&A text alone,0.818,0.125,0.008,16.255
10-K and 10-Q,0.869,0.149,0.008,19.368
10-K and 10-Q + MD&A text,0.904,0.152,0.008,19.690
+ NT,0.874,0.157,0.008,20.385
+ NT + MD&A text,0.900,0.148,0.008,19.250
+ NT + 8-K,0.866,0.153,0.008,19.818
+ NT + 8-K + MD&A text,0.894,0.162,0.008,21.058


## Checking the Text by Year and by Source

The MD&A comes from two places: EDGAR-CORPUS for most 10-Ks up to fiscal 2020, and our own cutting of the SEC filings for the rest. (A first run found that matching the corpus by filing year gave most 10-Ks the next year's text; the corpus is now matched by fiscal year with a check against future text.) Four checks:

0. Leakage: does the text of a 10-K filed before a bankruptcy already describe the bankruptcy?
1. The text score year by year: does it drop from 2021, when the source changes?
2. Within the same years (2016 to 2020), does the text score work as well on SEC-cut text as on corpus text?
3. Do the two kinds of text look different (length, how they start)?

In [8]:
df["text_source"] = df["text_adsh"].map(text_source)

ten_k_rows = df[df["form"].str.startswith("10-K") & df["text_adsh"].notna()]
print(pd.crosstab(ten_k_rows["year"], ten_k_rows["text_source"], normalize="index").round(2))

/var/folders/79/fbgym5f51g38htczmg5vvbh00000gp/T/ipykernel_77425/2448553065.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["text_source"] = df["text_adsh"].map(text_source)


text_source  EDGAR-CORPUS  SEC, our cut
year                                   
2009                 0.87          0.13
2010                 0.85          0.15
2011                 0.89          0.11
2012                 0.93          0.07
2013                 0.93          0.07
2014                 0.93          0.07
2015                 0.93          0.07
2016                 0.93          0.07
2017                 0.94          0.06
2018                 0.94          0.06
2019                 0.95          0.05
2020                 0.95          0.05
2021                 0.76          0.24
2022                 0.02          0.98
2023                 0.01          0.99
2024                 0.01          0.99
2025                 0.01          0.99


In [9]:
#0. leakage: how often the text already says the company filed for bankruptcy, for 10-Ks that went bankrupt within a
#year (and so were filed before it) and for the others. If the text is from the right 10-K, it should be rare for both
import re
filed = re.compile(r"filed (a )?voluntary petition|commenced (a )?(voluntary )?(case|cases|proceedings) under chapter 11|filed for (bankruptcy|chapter 11)", re.I)
check = ten_k_rows[ten_k_rows["text_adsh"] == ten_k_rows["adsh"]]
rows = []
for (name, label), part in check.groupby(["text_source", "Bankruptcy"]):
    part = part.sample(min(len(part), 400), random_state=1)
    rows.append({"text source": name, "went bankrupt": int(label), "checked": len(part),
                 "text says it filed for bankruptcy": np.mean([bool(filed.search(read_text(a, 400000))) for a in part["text_adsh"]])})
pd.DataFrame(rows).round(3)

,text source,went bankrupt,checked,text says it filed for bankruptcy
0,EDGAR-CORPUS,0,400,0.015
1,EDGAR-CORPUS,1,376,0.032
2,"SEC, our cut",0,400,0.008
3,"SEC, our cut",1,211,0.014


In [10]:
#1. the text score alone, year by year (the same walk forward as above)
text_scores = text_walk_forward()
by_year = []
for year in TEST_YEARS:
    s = text_scores[text_scores["year"] == year]
    s = s[s["score"].notna()]
    y = df.loc[s.index, "Bankruptcy"]
    sec_share = (df.loc[s.index, "text_source"] == "SEC, our cut").mean()
    by_year.append({"year": year, "reports": len(s), "bankruptcies": int(y.sum()), "ROC-AUC": roc_auc_score(y, s["score"]),
                    "PR-AUC": average_precision_score(y, s["score"]), "random": y.mean(), "share of SEC-cut text": sec_share})
pd.DataFrame(by_year).set_index("year").round(3)

,reports,bankruptcies,ROC-AUC,PR-AUC,random,share of SEC-cut text
year,,,,,,
2016,23754,232,0.746,0.066,0.010,0.067
2017,23197,164,0.803,0.035,0.007,0.060
2018,22516,164,0.855,0.066,0.007,0.058
2019,22027,247,0.852,0.147,0.011,0.052
2020,21382,155,0.846,0.190,0.007,0.051
2021,22490,60,0.796,0.147,0.003,0.167
2022,24896,191,0.773,0.036,0.008,0.978
2023,23547,310,0.778,0.054,0.013,0.986
2024,22104,192,0.851,0.056,0.009,0.990


In [11]:
#2. same years, two sources: 10-K reports filed 2016 to 2020, by where their text came from
s = text_scores[text_scores["year"].between(2016, 2020)].dropna()
rows = df.loc[s.index]
keep = rows["form"].str.startswith("10-K")
out = []
for name, part in rows[keep].groupby("text_source"):
    y, sc = part["Bankruptcy"], s.loc[part.index, "score"]
    out.append({"text source": name, "10-Ks": len(part), "bankruptcies": int(y.sum()), "ROC-AUC": roc_auc_score(y, sc),
                "PR-AUC": average_precision_score(y, sc), "random": y.mean(), "median text score": sc.median()})
pd.DataFrame(out).set_index("text source").round(4)

,10-Ks,bankruptcies,ROC-AUC,PR-AUC,random,median text score
text source,,,,,,
EDGAR-CORPUS,27137,235,0.8290,0.1426,0.0087,0.0786
"SEC, our cut",1651,19,0.7438,0.0951,0.0115,0.0713


In [12]:
#3. do they look different: length and how they start
def text_length(adsh):
    with gzip.open(mdna_files[adsh], "rt") as f:
        return len(f.read())


sample = ten_k_rows.groupby(["year", "text_source"]).head(200)
sample = sample.assign(length=[text_length(a) for a in sample["text_adsh"]])
print(sample.groupby(["text_source"])["length"].describe().round(0))
print(sample.groupby(["year", "text_source"])["length"].median().unstack().round(0))
for name in ["EDGAR-CORPUS", "SEC, our cut"]:
    adsh = sample.loc[sample["text_source"] == name, "text_adsh"].iloc[3]
    print("\n----", name, adsh)
    print(read_text(adsh, 600).replace("\n", " | "))

               count     mean      std     min      25%      50%       75%  \
text_source                                                                  
EDGAR-CORPUS  2692.0  52381.0  41336.0  3111.0  19350.0  45252.0   73489.0   
SEC, our cut  3066.0  91040.0  72836.0  2218.0  43312.0  74192.0  119089.0   

                   max  
text_source             
EDGAR-CORPUS  400000.0  
SEC, our cut  400000.0  
text_source  EDGAR-CORPUS  SEC, our cut
year                                   
2009              72966.0      101396.0
2010              82333.0       95940.0
2011              74664.0       85419.0
2012              37324.0       89521.0
2013              30297.0       88088.0
2014              26606.0       89180.0
2015              33850.0       87418.0
2016              36963.0       85111.0
2017              47170.0       85790.0
2018              44147.0       87452.0
2019              47966.0       78498.0
2020              42084.0       70975.0
2021              52528.0  